# Dynamic Systems Modeling
# UNIT 2 - PRACTICAL 1
## Newton vs Lagrange: Implementation and Comparison

---

**Instructor**: Dr. Sunny Nanade
**Duration**: 1 hour

---

## Objectives

1. Implement Newton's force-based method in Python
2. Implement Lagrange's energy-based method with SymPy
3. Compare both approaches for simple pendulum
4. Verify both give same equations of motion
5. Apply to double pendulum (where Lagrange shines!)

---

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sympy as sp
from sympy import symbols, Function, diff, simplify, cos, sin, solve, Eq

plt.rcParams['figure.figsize'] = (12, 8)
sp.init_printing(use_latex='mathjax')

print(' Libraries loaded')


---

# PART 1: Simple Pendulum - Newton's Method

## Manual Derivation

From Lecture 1, we derived:
$$\ddot{\theta} + \frac{g}{L}\sin\theta = 0$$

Let's implement and simulate:

---

In [ ]:
# Newton's method: Simple Pendulum

# Parameters
m = 1.0  # mass (kg)
L = 1.0  # length (m)
g = 9.8  # gravity (m/s²)

# Define ODE from Newton's derivation


def pendulum_newton(t, y):
    theta, omega = y
    dtheta_dt = omega
    domega_dt = -(g/L) * np.sin(theta)
    return [dtheta_dt, domega_dt]


# Initial conditions
theta0 = np.radians(30)  # 30 degrees
omega0 = 0.0

# Solve
t_span = [0, 10]
t_eval = np.linspace(0, 10, 500)
sol_newton = solve_ivp(pendulum_newton, t_span, [theta0, omega0],
                       t_eval=t_eval, method='RK45')

print(' Newton method: Simulation complete')
print(f' Equation: θ̈ = -(g/L)sin(θ) = -({g}/{L})sin(θ)')


---

# PART 2: Simple Pendulum - Lagrange with SymPy

## Symbolic Derivation

Use SymPy to derive equation automatically!

---

In [ ]:
# Lagrange method: Symbolic with SymPy

print('SYMBOLIC DERIVATION WITH SYMPY')
print('='*60)

# Define symbols
t = sp.Symbol('t', real=True, positive=True)
m_sym, g_sym, L_sym = sp.symbols('m g L', real=True, positive=True)
theta_sym = sp.Function('theta')(t)

# Kinetic Energy
T = sp.Rational(1, 2) * m_sym * L_sym**2 * sp.diff(theta_sym, t)**2
print(f'\nKinetic Energy: T = {T}')

# Potential Energy
V = -m_sym * g_sym * L_sym * sp.cos(theta_sym)
print(f'Potential Energy: V = {V}')

# Lagrangian
Lag = T - V
print(f'\nLagrangian: L = {sp.simplify(Lag)}')

# Euler-Lagrange
dL_dthetadot = sp.diff(Lag, sp.diff(theta_sym, t))
d_dt_dL_dthetadot = sp.diff(dL_dthetadot, t)
dL_dtheta = sp.diff(Lag, theta_sym)

EL_eq = d_dt_dL_dthetadot - dL_dtheta
print(f'\nEuler-Lagrange: {sp.simplify(EL_eq)} = 0')

# Solve for θ̈
theta_ddot = sp.solve(EL_eq, sp.diff(theta_sym, t, 2))[0]
print(f'\nEquation of Motion: θ̈ = {theta_ddot}')
print(f'Simplified: θ̈ = {sp.simplify(theta_ddot/L_sym)}')

print('\n Lagrange method: Same equation as Newton!')
print(' But derived automatically with SymPy!')


In [ ]:
# Both methods give same result - verify numerically

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Angle vs Time
ax = axes[0]
ax.plot(sol_newton.t, np.degrees(sol_newton.y[0]), 'b-',
        linewidth=2, label='Newton Method')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Angle (degrees)', fontsize=12)
ax.set_title(
    'Simple Pendulum: Newton vs Lagrange',
    fontsize=14,
    fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Phase Portrait
ax = axes[1]
ax.plot(np.degrees(sol_newton.y[0]), sol_newton.y[1], 'r-', linewidth=2)
ax.set_xlabel('Angle θ (degrees)', fontsize=12)
ax.set_ylabel('Angular Velocity ω (rad/s)', fontsize=12)
ax.set_title('Phase Portrait', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linestyle='--', linewidth=1)
ax.axvline(0, color='k', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

print('\n Both Newton and Lagrange methods verified!')
print(' Same physics, different mathematical approaches!')


---

# PART 3: Double Pendulum - Where Lagrange Shines!

## System Description

Two pendulums connected in series:
- Pendulum 1: mass $m_1$, length $L_1$, angle $\theta_1$
- Pendulum 2: mass $m_2$, length $L_2$, angle $\theta_2$

**Newton's method**: 4 coupled equations with 2 tension forces (messy!)

**Lagrange's method**: 2 clean equations, no tensions!

---

### Step-by-Step Procedure: Double Pendulum via Lagrange

> This section fills the missing derivation workflow before simulation.

**Step 1: Choose generalized coordinates**
- Use angles $\theta_1(t)$ and $\theta_2(t)$ measured from vertical for link 1 and link 2.
- Degrees of freedom: 2.

**Step 2: Write kinematics of both masses**
- Mass 1 position:
  $$x_1 = L_1\sin\theta_1, \quad y_1 = -L_1\cos\theta_1$$
- Mass 2 position:
  $$x_2 = x_1 + L_2\sin\theta_2, \quad y_2 = y_1 - L_2\cos\theta_2$$
- Differentiate to obtain velocities $\dot{x}_1, \dot{y}_1, \dot{x}_2, \dot{y}_2$.

**Step 3: Build total kinetic energy $T$**
- $$T = \frac{1}{2}m_1(\dot{x}_1^2+\dot{y}_1^2)+\frac{1}{2}m_2(\dot{x}_2^2+\dot{y}_2^2)$$
- Simplified form used in this notebook:
  $$T = \frac{1}{2}m_1L_1^2\dot{\theta}_1^2 + \frac{1}{2}m_2\left(L_1^2\dot{\theta}_1^2 + L_2^2\dot{\theta}_2^2 + 2L_1L_2\dot{\theta}_1\dot{\theta}_2\cos(\theta_1-\theta_2)\right)$$

**Step 4: Build total potential energy $V$**
- $$V = -m_1gL_1\cos\theta_1 - m_2g\left(L_1\cos\theta_1 + L_2\cos\theta_2\right)$$

**Step 5: Form Lagrangian**
- $$\mathcal{L}(\theta_1,\theta_2,\dot{\theta}_1,\dot{\theta}_2)=T-V$$

**Step 6: Apply Euler-Lagrange equation for each coordinate**
- For $\theta_1$:
  $$\frac{d}{dt}\left(\frac{\partial \mathcal{L}}{\partial \dot{\theta}_1}\right)-\frac{\partial \mathcal{L}}{\partial \theta_1}=0$$
- For $\theta_2$:
  $$\frac{d}{dt}\left(\frac{\partial \mathcal{L}}{\partial \dot{\theta}_2}\right)-\frac{\partial \mathcal{L}}{\partial \theta_2}=0$$
- This produces two coupled nonlinear second-order ODEs.

**Step 7: Convert to first-order state-space for numerical solver**
- State vector used in code:
  $$y=[\theta_1,\omega_1,\theta_2,\omega_2], \quad \omega_1=\dot{\theta}_1,\ \omega_2=\dot{\theta}_2$$
- Compute $\dot{\omega}_1$ and $\dot{\omega}_2$ from the coupled equations and return:
  $$[\dot{\theta}_1,\dot{\omega}_1,\dot{\theta}_2,\dot{\omega}_2]$$

**Step 8: Validate physically**
- Check trajectories and phase portraits.
- Verify sensitivity to initial conditions (chaotic behavior for larger motions).

**Why this matters:**
- Newton method requires tension-force bookkeeping at both joints.
- Lagrange method directly gives governing equations in the generalized coordinates, which is why it scales better for multi-DOF systems.

In [ ]:
# Double Pendulum with Lagrangian (manual - full symbolic takes too long)

print('DOUBLE PENDULUM: Lagrangian Approach')
print('='*60)

# The Lagrangian for double pendulum (pre-derived):
# T = (1/2)m₁L₁²θ̇₁² + (1/2)m₂[L₁²θ̇₁² + L₂²θ̇₂² + 2L₁L₂θ̇₁θ̇₂cos(θ₁-θ₂)]
# V = -m₁gL₁cos(θ₁) - m₂g[L₁cos(θ₁) + L₂cos(θ₂)]

# After applying Euler-Lagrange to both θ₁ and θ₂:
# Two coupled nonlinear ODEs (see textbook for full derivation)

# Parameters
m1, m2 = 1.0, 1.0
L1, L2 = 1.0, 1.0
g = 9.8


def double_pendulum(t, y):
    theta1, omega1, theta2, omega2 = y

    # Pre-compute common terms
    delta = theta2 - theta1
    den1 = (m1 + m2) * L1 - m2 * L1 * np.cos(delta)**2
    den2 = (L2 / L1) * den1

    # Equations from Lagrangian (derived from Euler-Lagrange)
    domega1_dt = (m2 * L1 * omega1**2 * np.sin(delta) * np.cos(delta) +
                  m2 * g * np.sin(theta2) * np.cos(delta) +
                  m2 * L2 * omega2**2 * np.sin(delta) -
                  (m1 + m2) * g * np.sin(theta1)) / den1

    domega2_dt = (-m2 * L2 * omega2**2 * np.sin(delta) * np.cos(delta) +
                  (m1 + m2) * g * np.sin(theta1) * np.cos(delta) -
                  (m1 + m2) * L1 * omega1**2 * np.sin(delta) -
                  (m1 + m2) * g * np.sin(theta2)) / den2

    return [omega1, domega1_dt, omega2, domega2_dt]


# Initial conditions: both at 45°, at rest
y0 = [np.radians(45), 0, np.radians(45), 0]

# Solve
t_span = [0, 20]
t_eval = np.linspace(0, 20, 2000)
sol_double = solve_ivp(
    double_pendulum,
    t_span,
    y0,
    t_eval=t_eval,
    method='RK45')

print('\n Double pendulum simulation complete')
print(' Lagrangian method handles complexity elegantly!')
print(' Newton method would require 4 equations + 2 tension unknowns!')


In [ ]:
# Visualize double pendulum motion

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Angles vs Time
ax = axes[0, 0]
ax.plot(
    sol_double.t,
    np.degrees(
        sol_double.y[0]),
    'b-',
    label='θ₁',
    linewidth=2)
ax.plot(
    sol_double.t,
    np.degrees(
        sol_double.y[2]),
    'r-',
    label='θ₂',
    linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Double Pendulum Angles', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Phase Portrait θ₁
ax = axes[0, 1]
ax.plot(np.degrees(sol_double.y[0]), sol_double.y[1], 'b-', linewidth=1.5)
ax.set_xlabel('θ₁ (degrees)')
ax.set_ylabel('ω₁ (rad/s)')
ax.set_title('Phase Portrait: Pendulum 1', fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Phase Portrait θ₂
ax = axes[1, 0]
ax.plot(np.degrees(sol_double.y[2]), sol_double.y[3], 'r-', linewidth=1.5)
ax.set_xlabel('θ₂ (degrees)')
ax.set_ylabel('ω₂ (rad/s)')
ax.set_title('Phase Portrait: Pendulum 2', fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 4: Trajectory of second mass
ax = axes[1, 1]
# Calculate positions
x1 = L1 * np.sin(sol_double.y[0])
y1 = -L1 * np.cos(sol_double.y[0])
x2 = x1 + L2 * np.sin(sol_double.y[2])
y2 = y1 - L2 * np.cos(sol_double.y[2])

ax.plot(x2, y2, 'g-', linewidth=0.5, alpha=0.7)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Trajectory of Mass 2 (Chaotic!)', fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.plot(0, 0, 'ko', markersize=8)

plt.tight_layout()
plt.show()

print('\n Double pendulum exhibits chaotic motion!')
print(' Lagrangian method made this problem tractable!')


---

# Summary

## What We Learned

### 1. Newton's Method Implementation
- Manual derivation required
- Direct implementation of $F = ma$
- Works well for simple systems

### 2. Lagrange's Method with SymPy
- **Symbolic derivation** automates the math!
- Define $T$ and $V$, SymPy does the rest
- No constraint forces to track

### 3. Key Comparison

| System | Newton | Lagrange |
|--------|--------|----------|
| Simple Pendulum | 2 equations + tension | 1 equation, no tension |
| Double Pendulum | 4 equations + 2 tensions | 2 equations, no tensions |

### 4. When to Use Each?

- **Newton**: Simple systems, need constraint forces
- **Lagrange**: Complex systems, multiple DOF, don't care about constraints

---

## Practical Exercises

**Exercise 1**: Modify simple pendulum with damping 
- Add term $-b\dot{\theta}$ to equation
- Observe oscillations decay

**Exercise 2**: Try different initial conditions for double pendulum
- Small angles: periodic motion
- Large angles: chaotic motion

**Exercise 3**: Implement spring-mass system both ways
- Newton: $F = -kx$
- Lagrange: $T = \frac{1}{2}m\dot{x}^2$, $V = \frac{1}{2}kx^2$

---

**End of Practical**

**Unit 2 Complete!** Next: Unit 3 - Rigid Body Kinematics

---